# 并发与任务队列

各节为独立示例，可按需阅读和运行。

目录：

- 并发和任务工具对比
- concurrent.futures
- threading.Lock 和 RLock
- threading.Semaphore 和 BoundedSemaphore
- threading.Event
- threading.Condition
- threading.Barrier
- threading.Timer
- threading.local
- queue.Queue
- multiprocessing.Pool
- multiprocessing.SimpleQueue
- multiprocessing.Pipe
- multiprocessing.Value 和 Array
- multiprocessing.Manager
- async 和 await
- asyncio 任务和取消
- asyncio.TaskGroup
- asyncio.Queue
- Celery
- Dramatiq
- ARQ
- Ray
- Dask

## 并发和任务工具对比

这个文件只保留横向选择信息，具体 API 示例分别放在同目录下的主题 notebook。

### 进程、线程、协程

| 技术 | 资源开销 | 并行能力 | 适用场景 | 典型用例 |
| --- | --- | --- | --- | --- |
| 进程 | 高 | 是，多核 | CPU 密集型、隔离性要求高 | 视频编码、科学计算 |
| 线程 | 中 | CPython 下受 GIL 限制 | I/O 密集型、简单数据共享 | 数据库查询、GUI 事件处理 |
| 协程 | 极低 | 单线程并发 | 超高并发 I/O、低延迟需求 | Web 服务、爬虫、聊天服务 |

### 线程和协程

| 维度 | Thread | Coroutine |
| --- | --- | --- |
| 调度机制 | 操作系统抢占式调度 | 用户态协作式调度 |
| 切换开销 | 较高 | 很低 |
| 内存占用 | 每个线程通常需要较大的栈 | 协程状态通常更小 |
| 编程重点 | 处理竞态条件和锁 | 避免阻塞事件循环 |

### 分布式和任务工具

| 框架 | 适合场景 | 核心特点 |
| --- | --- | --- |
| Celery | 稳定后台任务队列 | 生态成熟，broker/backend 模型清晰 |
| Ray | 分布式计算、模型推理、Actor 状态 | `@ray.remote` 把函数和类变成远程任务 |
| Dask | 分布式数据处理、任务图 | 适合 Pandas/NumPy 风格计算扩展 |
| Dramatiq | 轻量同步任务队列 | 比 Celery 简洁，支持 Redis/RabbitMQ |
| ARQ | asyncio 项目里的异步任务队列 | 完全 async/await，只依赖 Redis |

## concurrent.futures

`concurrent.futures` 提供统一的线程池和进程池接口。`ThreadPoolExecutor` 适合 I/O 密集任务，`ProcessPoolExecutor` 适合 CPU 密集任务。

In [ ]:
import concurrent.futures
import time


def fetch_data(x: int) -> str:
    print(f"Fetching data for {x}...")
    time.sleep(1)
    return f"Data {x}"


with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
    result = list(executor.map(fetch_data, [1, 2, 3, 4, 5]))

print(result)

In [ ]:
from concurrent.futures import ProcessPoolExecutor
from time import sleep


def compute_square(x: int) -> int:
    print(f"Computing square for {x}...")
    sleep(1)
    return x * x


if __name__ == "__main__":
    with ProcessPoolExecutor(max_workers=3) as executor:
        result = list(executor.map(compute_square, [1, 2, 3, 4, 5]))
    print(result)

## threading.Lock 和 RLock

`Lock` 用于普通互斥；`RLock` 是可重入锁，允许同一线程在嵌套调用中重复获取同一把锁。

### 选择规则

| 场景 | 推荐工具 |
| --- | --- |
| 简单临界区 | `Lock` |
| 同一线程会嵌套进入同一临界区 | `RLock` |
| 多把锁同时使用 | 固定获取顺序，必要时使用超时 |

In [ ]:
from threading import Lock, RLock, Thread
import time
from typing import Any


def process(source: Any, lock: Lock) -> None:
    with lock:
        print(f"Processing {source}")
        time.sleep(1)
        print(f"Finished processing {source}")


def save_result(source: str, lock: RLock) -> None:
    with lock:
        print(f"Save result for {source}")


def process_data(source: str, lock: RLock) -> None:
    with lock:
        print(f"Process {source}")
        save_result(source, lock)


lock = Lock()
thread = Thread(target=process, args=("Data Source", lock))
thread.start()
thread.join()

rlock = RLock()
thread = Thread(target=process_data, args=("critical data", rlock))
thread.start()
thread.join()

## threading.Semaphore 和 BoundedSemaphore

`Semaphore` 用于限制同时进入某个资源区的线程数量。`BoundedSemaphore` 会检查释放次数，适合防止错误地多次 `release()`。

In [ ]:
import random
import threading
import time


semaphore = threading.Semaphore(3)


def access_resource(thread_id: int) -> None:
    with semaphore:
        print(f"Thread-{thread_id} acquired the resource")
        time.sleep(random.uniform(0.2, 0.5))
        print(f"Thread-{thread_id} released the resource")


threads = [
    threading.Thread(target=access_resource, args=(index,)) for index in range(5)
]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()

bounded = threading.BoundedSemaphore(3)
try:
    bounded.release()
except ValueError as exc:
    print(f"BoundedSemaphore prevents over-release: {exc}")

## threading.Event

`Event` 用于在线程之间传递“一次性通知”。一个线程调用 `set()` 后，等待中的线程会继续执行；需要重复使用时可以调用 `clear()` 重置状态。

In [ ]:
from threading import Event, Thread
import time


def countdown(number: int, event: Event) -> None:
    print(f"Countdown from {number} started")
    for index in range(number):
        print(f"Countdown: {number - index}")
        time.sleep(1)
    print(f"Countdown from {number} finished")
    event.set()


event = Event()
thread = Thread(target=countdown, args=(5, event))
thread.start()

event.wait()
print("Countdown completed, main thread continues")
event.clear()

## threading.Condition

`Condition` 把锁和等待队列组合在一起，适合生产者消费者这类“条件未满足就等待，条件满足后通知”的协作场景。

In [ ]:
from threading import Condition, Thread
import time


def produce(condition: Condition, items: list[int]) -> None:
    for item in range(5):
        with condition:
            items.append(item)
            print(f"Produced {item}")
            condition.notify()
        time.sleep(2)


def consume(condition: Condition, items: list[int]) -> None:
    while True:
        with condition:
            while not items:
                print("等待生产...")
                condition.wait()
            item = items.pop(0)
            print(f"Consumed {item}")
            if item == 4:
                break


items: list[int] = []
condition = Condition()
producer_thread = Thread(target=produce, args=(condition, items))
consumer_thread = Thread(target=consume, args=(condition, items))
producer_thread.start()
consumer_thread.start()
producer_thread.join()
consumer_thread.join()

## threading.Barrier

`Barrier` 让固定数量的线程在同一个阶段边界等待，直到所有线程都到达后再一起进入下一阶段。

In [ ]:
import random
import threading
import time


phase1_barrier = threading.Barrier(3, action=lambda: print("开始清洗"))
phase2_barrier = threading.Barrier(3, action=lambda: print("开始分析"))


def data_processing(thread_id: int) -> None:
    load_time = random.uniform(0.2, 0.5)
    time.sleep(load_time)
    print(f"线程 {thread_id} 数据加载完成")
    phase1_barrier.wait()

    clean_time = random.uniform(0.2, 0.5)
    time.sleep(clean_time)
    print(f"线程 {thread_id} 数据清洗完成")
    phase2_barrier.wait()

    print(f"线程 {thread_id} 数据分析完成")


threads = [
    threading.Thread(target=data_processing, args=(index,)) for index in range(1, 4)
]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()

## threading.Timer

`Timer` 用于延迟启动一个函数调用。本质上它仍然会创建线程，适合轻量延迟任务，不适合复杂任务调度。

In [ ]:
import threading
import time
from typing import Any


def do_something(name: Any, queue: list[Any]) -> None:
    queue.append(name)


pool: list[Any] = []
timer = threading.Timer(2, do_something, args=("Alice", pool))
timer.start()

while not pool:
    print("Waiting for Alice...")
    time.sleep(1)

print(pool)

## threading.local

`threading.local()` 用于保存线程局部状态。每个线程看到的是自己的属性值，适合保存请求上下文、连接上下文等线程内状态。

In [ ]:
import random
import threading
import time


local_data = threading.local()


def thread_task(thread_id: int) -> None:
    local_data.value = thread_id
    time.sleep(random.randrange(1, 3))
    print(f"线程 {thread_id} 的局部数据: {local_data.value}")


threads = [threading.Thread(target=thread_task, args=(index,)) for index in range(5)]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()

## queue.Queue

`queue.Queue` 是线程间通信队列。常见变体包括 FIFO 队列、`LifoQueue` 后进先出队列和 `PriorityQueue` 优先级队列。

In [ ]:
import queue
import threading
import time


def produce(production: queue.LifoQueue[int | None]) -> None:
    for goods in range(5):
        time.sleep(0.2)
        print(f"生产者生产了数据: {goods}")
        production.put(goods)


def consume(production: queue.LifoQueue[int | None]) -> None:
    while True:
        item = production.get()
        if item is None:
            break
        print(f"消费者消费了数据: {item}")
        production.task_done()


production: queue.LifoQueue[int | None] = queue.LifoQueue()
producer_thread = threading.Thread(target=produce, args=(production,))
consumer_thread = threading.Thread(target=consume, args=(production,))
producer_thread.start()
consumer_thread.start()
producer_thread.join()
production.put(None)
consumer_thread.join()

### 队列类型对比

- `queue.Queue()` 是线程队列。
- `multiprocessing.Queue()` 是进程间队列。
- `multiprocessing.SimpleQueue()` 是进程间简单队列。
- `multiprocessing.Manager().Queue()` 是进程间共享队列。
- `asyncio.Queue()` 是协程队列。

## multiprocessing.Pool

`Pool` 是进程池接口，适合把同一个函数并行应用到多个输入上。

| 方法 | 说明 |
| --- | --- |
| `map()` | 按顺序返回所有结果 |
| `imap_unordered()` | 逐个返回结果，顺序不固定 |
| `apply_async()` | 异步提交单个任务 |
| `close()` / `join()` | 停止接收新任务并等待完成 |

In [ ]:
import multiprocessing
import time


def square(x: int) -> int:
    time.sleep(1)
    return x * x


if __name__ == "__main__":
    pool = multiprocessing.Pool(processes=4)
    numbers = [1, 2, 3, 4, 5]
    result = pool.imap_unordered(square, numbers)
    for item in result:
        print(item)
    pool.close()
    pool.join()

## multiprocessing.SimpleQueue

`SimpleQueue` 是进程间简单队列，接口比 `Queue` 少，适合基本的生产者消费者通信。

In [ ]:
import multiprocessing
import time


def producer(q) -> None:
    for item in range(5):
        time.sleep(0.2)
        print(f"Producer putting item {item} into queue")
        q.put(item)


def consumer(q) -> None:
    for _ in range(5):
        item = q.get()
        print(f"Consumer got item {item} from queue")
        time.sleep(0.4)


if __name__ == "__main__":
    q = multiprocessing.SimpleQueue()
    producer_process = multiprocessing.Process(target=producer, args=(q,))
    consumer_process = multiprocessing.Process(target=consumer, args=(q,))
    producer_process.start()
    consumer_process.start()
    producer_process.join()
    consumer_process.join()

## multiprocessing.Pipe

`Pipe` 用于两个进程之间双向通信。它返回一对连接对象，双方可以通过 `send()` 和 `recv()` 传递可序列化对象。

In [ ]:
import multiprocessing


def worker(pipe) -> None:
    data = pipe.recv()
    print(f"Received: {data}")
    pipe.send(f"Processed {data}")


def main() -> None:
    parent, child = multiprocessing.Pipe()
    process = multiprocessing.Process(target=worker, args=(child,))
    process.start()
    parent.send("Data from main process")
    print(f"Received from worker: {parent.recv()}")
    process.join()


if __name__ == "__main__":
    multiprocessing.set_start_method("spawn")
    main()

## multiprocessing.Value 和 Array

`Value` 用于共享单个标量，`Array` 用于共享固定长度数组。复合操作不是原子的，即使启用了自动锁，也应该在读改写时显式获取锁。

| 工具 | 适合场景 |
| --- | --- |
| `Value` | 跨进程共享单个整数、浮点数等标量 |
| `RawValue` | 不带自动同步的底层标量共享 |
| `Array` | 跨进程共享固定长度数组 |
| `RawArray` | 不带自动同步的底层数组共享 |

In [ ]:
import multiprocessing
import time


def increment_value(shared_value) -> None:
    for _ in range(5):
        time.sleep(0.2)
        with shared_value.get_lock():
            shared_value.value += 1
            print(f"Shared value in process: {shared_value.value}")


def increment_array(shared_array, index: int) -> None:
    for _ in range(5):
        time.sleep(0.2)
        with shared_array.get_lock():
            shared_array[index] += 1
            print(f"Shared array[{index}] in process: {shared_array[index]}")


if __name__ == "__main__":
    shared_value = multiprocessing.Value("i", 0)
    value_processes = [
        multiprocessing.Process(target=increment_value, args=(shared_value,))
        for _ in range(3)
    ]
    for process in value_processes:
        process.start()
    for process in value_processes:
        process.join()
    print(f"Final shared value: {shared_value.value}")

    shared_array = multiprocessing.Array("i", [0, 0, 0], lock=True)
    array_processes = [
        multiprocessing.Process(target=increment_array, args=(shared_array, index))
        for index in range(3)
    ]
    for process in array_processes:
        process.start()
    for process in array_processes:
        process.join()
    print(f"Final shared array: {shared_array[:]}")

## multiprocessing.Manager

`Manager` 提供可在进程间共享的容器对象，例如列表和字典。它比共享内存对象更灵活，但也更重。

In [ ]:
import multiprocessing
import random
import time


def work(shared_list, item: int) -> None:
    time.sleep(random.randint(1, 3))
    print(f"I am {item} process, current list: {shared_list}")
    shared_list.append(item)


if __name__ == "__main__":
    shared_list = multiprocessing.Manager().list()
    processes = [
        multiprocessing.Process(target=work, args=(shared_list, index))
        for index in range(5)
    ]
    for process in processes:
        process.start()
    for process in processes:
        process.join()
    print(f"Final list: {shared_list}")

## async 和 await

`asyncio` 使用协程处理高并发 I/O。`async def` 定义协程函数，`await` 把控制权交还给事件循环，`asyncio.run()` 启动顶层协程。

In [ ]:
import asyncio


async def hello_world() -> None:
    print("Hello")
    await asyncio.sleep(1)
    print("World")


async def main() -> None:
    await hello_world()


asyncio.run(main())

In [ ]:
import asyncio


async def fetch_data(x: int) -> str:
    print(f"Fetching data for {x}...")
    await asyncio.sleep(1)
    return f"Data {x}"


async def main() -> None:
    tasks = [fetch_data(index) for index in range(5)]
    results = await asyncio.gather(*tasks)
    print(results)


asyncio.run(main())

In [ ]:
import asyncio


async def task(name: str, delay: int) -> str:
    if delay == 2:
        raise ValueError(f"Task {name} encountered an error")
    await asyncio.sleep(delay)
    return f"Task {name} completed"


async def main() -> None:
    tasks = [task("A", 1), task("B", 2), task("C", 3)]
    try:
        results = await asyncio.gather(*tasks)
        print(results)
    except Exception as exc:
        print(f"Error occurred: {exc}")


asyncio.run(main())

## asyncio 任务和取消

`asyncio.create_task()` 把协程包装成任务并交给事件循环调度。长期运行的任务要正确处理取消信号。

In [ ]:
import asyncio


async def timed_task(name: str, delay: float) -> None:
    print(f"Task {name} started")
    await asyncio.sleep(delay)
    print(f"Task {name} completed after {delay} seconds")


async def main() -> None:
    task_a = asyncio.create_task(timed_task("A", 2), name="task-a")
    task_b = asyncio.create_task(timed_task("B", 1), name="task-b")
    await asyncio.gather(task_a, task_b)


asyncio.run(main())

In [ ]:
import asyncio


async def heartbeat() -> None:
    try:
        while True:
            print("Running...")
            await asyncio.sleep(1)
    except asyncio.CancelledError:
        print("Stopped")
        raise


async def main() -> None:
    task = asyncio.create_task(heartbeat())
    await asyncio.sleep(3)
    task.cancel()
    try:
        await task
    except asyncio.CancelledError:
        pass


asyncio.run(main())

## asyncio.TaskGroup

`TaskGroup` 是 Python 3.11+ 的结构化并发工具。它适合“一组任务必须在同一作用域内一起完成”的场景；如果某个任务失败，任务组会取消其他未完成任务。

In [ ]:
import asyncio


async def fetch_user() -> dict[str, str]:
    await asyncio.sleep(0.2)
    return {"id": "U-001", "name": "Ada"}


async def fetch_orders() -> list[str]:
    await asyncio.sleep(0.3)
    return ["ORD-001", "ORD-002"]


async def main() -> None:
    async with asyncio.TaskGroup() as group:
        user_task = group.create_task(fetch_user())
        orders_task = group.create_task(fetch_orders())

    print(user_task.result())
    print(orders_task.result())


asyncio.run(main())

## asyncio.Queue

`asyncio.Queue` 是协程之间的生产者消费者队列。它只能在事件循环内使用，不能替代线程队列或进程队列。

In [ ]:
import asyncio


async def producer(queue: asyncio.Queue[int | None]) -> None:
    for item in range(5):
        print(f"Producer: producing {item}")
        await queue.put(item)
        await asyncio.sleep(1)


async def consumer(queue: asyncio.Queue[int | None]) -> None:
    while True:
        item = await queue.get()
        if item is None:
            break
        print(f"Consumer: consumed {item}")
        queue.task_done()
        await asyncio.sleep(2)


async def main() -> None:
    queue: asyncio.Queue[int | None] = asyncio.Queue(maxsize=3)
    producer_task = asyncio.create_task(producer(queue))
    consumer_task = asyncio.create_task(consumer(queue))
    await producer_task
    await queue.put(None)
    await consumer_task


asyncio.run(main())

## Celery

Celery 适合稳定的后台任务系统，例如发送邮件、报表生成、视频转码、异步 Webhook 和批量同步。生产里通常拆成 worker 和 producer 两类进程。

- broker: Redis、RabbitMQ 等消息中间件。
- backend: 保存任务结果，常用 Redis 或数据库。
- worker: 执行任务。
- producer: 提交任务。

```zsh
celery -A learnpy.workers.celery.tasks worker --loglevel=info
python -m learnpy.workers.celery.client
```

In [ ]:
import os
from typing import Final

from celery import Celery

DEFAULT_REDIS_URL: Final[str] = "redis://localhost:6379/0"
REDIS_URL: Final[str] = os.getenv("TASK_QUEUE_REDIS_URL", DEFAULT_REDIS_URL)

app = Celery("task_queue_examples", broker=REDIS_URL, backend=REDIS_URL)
app.conf.update(
    task_serializer="json",
    result_serializer="json",
    accept_content=["json"],
    timezone="Asia/Shanghai",
    enable_utc=False,
    task_track_started=True,
    task_time_limit=30,
    result_expires=3600,
)


@app.task(name="notebook.celery.add")
def celery_add(x: int, y: int) -> int:
    return x + y


@app.task(name="notebook.celery.multiply")
def celery_multiply(x: int, y: int) -> int:
    return x * y

### Canvas 工作流

Celery Canvas 可以组合任务：`chain()` 串行执行，`group()` 并行执行，`chord()` 在一组任务完成后执行汇总任务。

In [ ]:
from celery import chain, chord, group

from learnpy.workers.celery.tasks import add, multiply, sum_numbers

single = add.apply_async(args=(4, 6), countdown=1)
print("Task ID:", single.id)

chained = chain(add.s(2, 3), multiply.s(10)).apply_async()
grouped = group(add.s(index, index + 1) for index in range(3)).apply_async()
summarized = chord(
    [multiply.s(index, 2) for index in range(1, 4)], sum_numbers.s()
).apply_async()

print("Chain:", chained.get(timeout=10))
print("Group:", grouped.get(timeout=10))
print("Chord:", summarized.get(timeout=10))

## Dramatiq

Dramatiq API 比 Celery 更轻，适合 Redis/RabbitMQ 上的后台任务。常用能力包括 actor、重试、结果存储和 pipeline。

```zsh
dramatiq learnpy.workers.dramatiq.tasks
python -m learnpy.workers.dramatiq.client
```

In [ ]:
import os
from typing import Final

import dramatiq
from dramatiq.brokers.redis import RedisBroker
from dramatiq.results import Results
from dramatiq.results.backends import RedisBackend

DEFAULT_REDIS_URL: Final[str] = "redis://localhost:6379/0"
REDIS_URL: Final[str] = os.getenv("TASK_QUEUE_REDIS_URL", DEFAULT_REDIS_URL)

result_backend = RedisBackend(url=REDIS_URL)
redis_broker = RedisBroker(url=REDIS_URL)
redis_broker.add_middleware(Results(backend=result_backend))
dramatiq.set_broker(redis_broker)


@dramatiq.actor(
    actor_name="notebook_dramatiq_add",
    max_retries=3,
    min_backoff=1_000,
    store_results=True,
)
def dramatiq_add(x: int, y: int) -> int:
    return x + y

In [ ]:
import dramatiq

from learnpy.workers.dramatiq.tasks import add, multiply

message = add.send(3, 5)
print("Message ID:", message.message_id)
print("Result:", message.get_result(block=True, timeout=10_000))

workflow = dramatiq.pipeline([add.message(2, 3), multiply.message(10)]).run()
print("Pipeline Result:", workflow.get_result(block=True, timeout=10_000))

## ARQ

ARQ 基于 asyncio 和 Redis，适合异步 I/O 任务。任务函数是协程，天然适配 HTTP 请求、数据库访问等异步资源。

```zsh
arq learnpy.workers.arq.tasks.WorkerSettings
python -m learnpy.workers.arq.client
```

In [ ]:
from typing import Any

from arq import cron, func
from arq.connections import RedisSettings
from arq.worker import Retry


async def add(ctx: dict[str, Any], a: int, b: int) -> int:
    return a + b


async def flaky_job(ctx: dict[str, Any], value: int) -> int:
    if ctx["job_try"] < 3:
        raise Retry(defer=1)
    return value * 2


async def heartbeat(ctx: dict[str, Any]) -> None:
    print("ARQ worker heartbeat")


class WorkerSettings:
    functions = [
        func(add, name="add", keep_result=3600, timeout=10),
        func(flaky_job, name="flaky_job", keep_result=3600, timeout=15, max_tries=3),
    ]
    cron_jobs = [cron(heartbeat, minute={0, 30}, run_at_startup=True)]
    redis_settings = RedisSettings(host="localhost", port=6379, database=0)

In [ ]:
import asyncio

from arq import create_pool

from learnpy.workers.arq.tasks import redis_config_from_env


async def main() -> None:
    redis = await create_pool(redis_config_from_env().to_settings())
    try:
        job = await redis.enqueue_job("add", 3, 7)
        retried = await redis.enqueue_job("flaky_job", 21)
        print("Job ID:", job.job_id)
        print("Add Result:", await job.result(timeout=5))
        print("Retry Result:", await retried.result(timeout=10))
    finally:
        await redis.close()


asyncio.run(main())

## Ray

Ray 适合 CPU/GPU 计算、模型推理、批处理和 actor 状态管理。它不是传统任务队列，核心能力是把 Python 函数和类变成可分布式调度的任务。

```zsh
ray start --head --port=6379
ray start --address='192.168.1.100:6379'
```

In [ ]:
import ray

ray.init(ignore_reinit_error=True, include_dashboard=False)


@ray.remote(num_cpus=1)
def score_user(user_id: int) -> dict[str, int]:
    return {"user_id": user_id, "score": user_id * 10}


@ray.remote
class Counter:
    def __init__(self) -> None:
        self.value = 0

    def increment(self, step: int = 1) -> int:
        self.value += step
        return self.value


scores = ray.get([score_user.remote(user_id) for user_id in range(5)])
print(scores)

counter = Counter.remote()
print(ray.get([counter.increment.remote() for _ in range(3)]))

ray.shutdown()

## Dask

Dask 适合把 Pandas/NumPy 风格的数据计算扩展到多核或多机，也适合构建延迟执行图。它和 Ray 都能做分布式计算，但 Dask 更偏数据分析和任务图。

In [ ]:
from dask import delayed
from dask.distributed import Client, LocalCluster, as_completed


def load_order(order_id: int) -> dict[str, int]:
    return {"order_id": order_id, "amount": order_id * 100}


def discount(order: dict[str, int]) -> int:
    return int(order["amount"] * 0.9)


with LocalCluster(n_workers=2, threads_per_worker=1, dashboard_address=None) as cluster:
    with Client(cluster) as client:
        orders = [delayed(load_order)(order_id) for order_id in range(1, 6)]
        amounts = [delayed(discount)(order) for order in orders]
        total = delayed(sum)(amounts)
        print("Delayed Total:", total.compute())

        futures = client.map(load_order, range(6, 9))
        for future in as_completed(futures):
            print("Future Result:", future.result())

### 远程集群

| 角色 | IP 地址 |
| --- | --- |
| Scheduler | 192.168.1.10 |
| Worker 1 | 192.168.1.11 |
| Worker 2 | 192.168.1.12 |

Scheduler 节点运行：

```zsh
dask-scheduler
```

Worker 节点运行：

```zsh
dask-worker tcp://192.168.1.10:8786
```

客户端连接：

```python
from dask.distributed import Client

client = Client('tcp://192.168.1.10:8786')
```